## Setup

In [41]:
import numpy as np
import pandas as pd
import pickle
import random

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

# For experiment analysis (keeping parallel to andrew's baseline)
import itertools
import statistics
import matplotlib.pyplot as plt
import seaborn as sns

import exp as exptb

In [42]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# "label" for comparability and comparability with andrew's encoded pipeline.
ENCODING_MODE = "label"   # onehot for future upgrade.
BATCH_SIZE = 512
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 6


## Dataset Analysis & preprocessing

In [43]:
# Load and begin prepping strip search data
df = pd.read_csv('./datasets/torontostripsearch.csv', delimiter=',');df.head()

,Arrest_Year,Arrest_Month,EventID,ArrestID,PersonID,Perceived_Race,Sex,Age_group__at_arrest_,Youth_at_arrest__under_18_years,ArrestLocDiv,...,Actions_at_arrest___Resisted__d,Actions_at_arrest___Mental_inst,Actions_at_arrest___Assaulted_o,Actions_at_arrest___Cooperative,SearchReason_CauseInjury,SearchReason_AssistEscape,SearchReason_PossessWeapons,SearchReason_PossessEvidence,ItemsFound,ObjectId
0,2020,July-Sept,1005907,6017884.0,326622,White,M,Aged 35 to 44 years,Not a youth,54,...,0,0,0,1,NaN,NaN,NaN,NaN,NaN,1
1,2020,July-Sept,1014562,6056669.0,326622,White,M,Aged 35 to 44 years,Not a youth,54,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,2
2,2020,Oct-Dec,1029922,6057065.0,326622,Unknown or Legacy,M,Aged 35 to 44 years,Not a youth,54,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,3
3,2021,Jan-Mar,1052190,6029059.0,327535,Black,M,Aged 25 to 34 years,Not a youth,XX,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,4
4,2021,Jan-Mar,1015512,6040372.0,327535,South Asian,M,Aged 25 to 34 years,Not a youth,XX,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,5


In [44]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()}") #list unique valuyes for each column to get a sense of the data.
    print(df[col].unique())


Arrest_Year: 2
[2020 2021]
Arrest_Month: 4
['July-Sept' 'Oct-Dec' 'Jan-Mar' 'Apr-June']
EventID: 60003
[1005907 1014562 1029922 ... 1021067 1008998 1033395]
ArrestID: 64805
[6017884. 6056669. 6057065. ... 6064396. 6008662. 6032145.]
PersonID: 37347
[326622 327535 330778 ... 324057 331870 310583]
Perceived_Race: 8
['White' 'Unknown or Legacy' 'Black' 'South Asian' 'Indigenous'
 'Middle-Eastern' 'Latino' 'East/Southeast Asian' nan]
Sex: 3
['M' 'F' 'U']
Age_group__at_arrest_: 9
['Aged 35 to 44 years' 'Aged 25 to 34 years' 'Aged 45 to 54 years'
 'Aged 55 to 64 years' 'Aged 18 to 24 years' 'Aged 65 and older'
 'Aged 65 years and older' nan 'Aged 17 years and younger'
 'Aged 17 years and under']
Youth_at_arrest__under_18_years: 3
['Not a youth' 'Youth (aged 17 and younger)'
 'Youth (aged 17 years and under)']
ArrestLocDiv: 18
['54' 'XX' '42' '52' '14' '51' '53' '31' '11' '12' '13' '41' '22' '55'
 '43' '23' '33' '32']
StripSearch: 2
[0 1]
Booked: 2
[1 0]
Occurrence_Category: 31
['Assault & Ot

In [45]:

df['IsYouth'] = np.where(df['Youth_at_arrest__under_18_years'] == "Not a youth", 0, 1)
df = df.rename(columns={"Arrest_Month" : "Arrest_Quarter"})
df.drop(columns=df.columns.intersection(['SearchReason_CauseInjury', 'SearchReason_AssistEscape', 'SearchReason_PossessWeapons', 'Arrest_Year',
                                         'SearchReason_PossessEvidence', 'Youth_at_arrest__under_18_years', "_defensive_or_escape_risk",
                                         'ObjectId', 'EventID', 'ArrestID', 'PersonID', 'Booked', 'ItemsFound']), axis=1, inplace=True)

df.replace({'Perceived_Race': {np.nan: 'Unknown or Legacy'}}, inplace=True)

df = df.drop(df[df['Sex'] == 'U'].index)

df.dropna(how="any", inplace=True)
categorical_features = ['Perceived_Race', 'Sex', 'Occurrence_Category', 'ArrestLocDiv']
ordinal_features = ['Age_group__at_arrest_', 'Arrest_Quarter']
binary_features = ['IsYouth','Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__',
                  'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Mental_inst',
                  'Actions_at_arrest___Assaulted_o', 'Actions_at_arrest___Cooperative']


X = df.drop('StripSearch', axis=1)
X = X.drop('IsYouth', axis=1) # keep IsYouth and change how it affects the model in future work
y = df['StripSearch']

# Map the age groups to the specified values
custom_mapping = {
    'Aged 17 years and under': 0,
    'Aged 17 years and younger': 0,
    'Aged 18 to 24 years': 1,
    'Aged 25 to 34 years': 2,
    'Aged 35 to 44 years': 3,
    'Aged 45 to 54 years': 4,
    'Aged 55 to 64 years': 5,
    'Aged 65 and older': 6,
    'Aged 65 years and older': 6
}

quarter_mapping = {
"Jan-Mar" : 0,
"Apr-June" : 1,
"July-Sept" : 3,
"Oct-Dec" : 4
}

X['Age_group__at_arrest_'] = X['Age_group__at_arrest_'].map(custom_mapping)
X['Arrest_Quarter'] = X['Arrest_Quarter'].map(quarter_mapping)

# Store the reverse mappings for later use in analysis and visualization
rev_custom_mapping = {
    0 : 'Aged 17 years and younger',
    1 : 'Aged 18 to 24 years'      ,
    2 : 'Aged 25 to 34 years'      ,
    3 : 'Aged 35 to 44 years'      ,
    4 : 'Aged 45 to 54 years'      ,
    5 : 'Aged 55 to 64 years'      ,
    6 : 'Aged 65 and older'        
}

rev_quarter_mapping = {
    0 : "Jan-Mar"   ,
    1 : "Apr-June"  ,
    3 : "July-Sept" ,
    4 : "Oct-Dec"   
}

custom_encoders = {
    'Age_group__at_arrest_': rev_custom_mapping,
    'Arrest_Quarter': rev_quarter_mapping
}

In [46]:
X.head()

,Arrest_Quarter,Perceived_Race,Sex,Age_group__at_arrest_,ArrestLocDiv,Occurrence_Category,Actions_at_arrest___Concealed_i,Actions_at_arrest___Combative__,Actions_at_arrest___Resisted__d,Actions_at_arrest___Mental_inst,Actions_at_arrest___Assaulted_o,Actions_at_arrest___Cooperative
0,3,White,M,3,54,Assault & Other crimes against persons,0,0,0,0,0,1
1,3,White,M,3,54,Assault & Other crimes against persons,0,0,0,0,0,0
2,4,Unknown or Legacy,M,3,54,Assault & Other crimes against persons,0,0,0,0,0,0
3,0,Black,M,2,XX,Harassment/Threatening,0,0,0,0,0,0
4,0,South Asian,M,2,XX,FTA/FTC/Compliance Check/Parollee,0,0,0,0,0,0


In [47]:
# train/test split (same as Andrew)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Build analysis copies with label encoding so slicing/explanation-table code still works
X_train_analysis = X_train.copy()
X_test_analysis = X_test.copy()

#Analysing which encoders we require for our Dataset and NN baseline
# 1. We have 4 categorical features: Perceived_Race, Sex, Occurrence_Category, ArrestLocDiv
# 2. We have 2 ordinal features: Age_group__at_arrest_, Arrest
# 3. We have 7 binary features: IsYouth, Actions_at_arrest___Concealed_i, Actions_at_arrest___Combative__, Actions_at_arrest___Resisted__d, Actions_at_arrest___Mental_inst, Actions_at_arrest___Assaulted_o, Actions_at_arrest___Cooperative

# Output directories for NN baseline artifacts
import os
OUT_BASE_DIR = "baseline_nn_output"
ENCODER_DIR = os.path.join(OUT_BASE_DIR, "encoder_tables")
FP_DIR = os.path.join(OUT_BASE_DIR, "fp_analysis")
DICE_DIR = os.path.join(OUT_BASE_DIR, "dice_cf")
RECOURSE_TEMP_DIR = os.path.join(OUT_BASE_DIR, "recourse_temp")
RECOURSE_TABLE_DIR = os.path.join(OUT_BASE_DIR, "recourse_tables")
for _d in [ENCODER_DIR, FP_DIR, DICE_DIR, RECOURSE_TEMP_DIR, RECOURSE_TABLE_DIR]:
    os.makedirs(_d, exist_ok=True)
print("Output dirs ready under:", OUT_BASE_DIR)



Output dirs ready under: baseline_nn_output


## Encoder Analysis
 1) categorical features (label encoder), ordinal features (ordinal encoder), binary features (passthrough)
 2) categorical features (one-hot), ordinal features (ordinal encoder), binary features (passthrough)
 3) categorical features (one-hot), ordinal features (ordinal encoder), binary features (one-hot )


In [48]:
# Base split copies (avoid accidental in-place mutation across cases)
X_train_base = X_train.copy(deep=True)
X_test_base = X_test.copy(deep=True)

# Labels once (shared across all cases)
y_train_np = y_train.to_numpy(dtype=np.float32)
y_test_np = y_test.to_numpy(dtype=np.float32)


def build_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_ord():
    # Safer than default: does not crash on unseen test categories
    return OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# Keep only columns that are actually present in model frame (e.g., IsYouth may be dropped upstream)
cat_cols = [c for c in categorical_features if c in X_train_base.columns]
ord_cols = [c for c in ordinal_features if c in X_train_base.columns]
bin_cols = [c for c in binary_features if c in X_train_base.columns]

missing_cat = sorted(set(categorical_features) - set(cat_cols))
missing_ord = sorted(set(ordinal_features) - set(ord_cols))
missing_bin = sorted(set(binary_features) - set(bin_cols))

if missing_cat or missing_ord or missing_bin:
    print(" Skipped missing columns ->")
    print("  categorical:", missing_cat)
    print("  ordinal:", missing_ord)
    print("  binary:", missing_bin)

def run_case(case_name, preprocessor, Xtr, Xte):
    Xtr_enc = preprocessor.fit_transform(Xtr)
    Xte_enc = preprocessor.transform(Xte)

    scaler_local = StandardScaler()
    Xtr_np = scaler_local.fit_transform(Xtr_enc).astype(np.float32)
    Xte_np = scaler_local.transform(Xte_enc).astype(np.float32)

    print(case_name)
    print("  X_train:", Xtr_np.shape)
    print("  X_test :", Xte_np.shape)
    print("  Positive class ratio (train):", y_train_np.mean())

    return {
        "name": case_name,
        "preprocessor": preprocessor,
        "scaler": scaler_local,
        "X_train_np": Xtr_np,
        "X_test_np": Xte_np,
        "y_train_np": y_train_np,
        "y_test_np": y_test_np,
    }


# Case 1: encoded categorical + ordinal-encoded ordinal + binary passthrough
preprocessor_c1 = ColumnTransformer(
    transformers=[
        ("cat_ordenc", build_ord(), cat_cols),
        ("ord_ordenc", build_ord(), ord_cols),
        ("bin_pass", "passthrough", bin_cols),
    ],
    remainder="drop",
)

# Case 2: one-hot categorical + ordinal-encoded ordinal + binary passthrough
preprocessor_c2 = ColumnTransformer(
    transformers=[
        ("cat_ohe", build_ohe(), cat_cols),
        ("ord_ordenc", build_ord(), ord_cols),
        ("bin_pass", "passthrough", bin_cols),
    ],
    remainder="drop",
)

# Case 3: one-hot categorical + ordinal-encoded ordinal + one-hot binary
preprocessor_c3 = ColumnTransformer(
    transformers=[
        ("cat_ohe", build_ohe(), cat_cols),
        ("ord_ordenc", build_ord(), ord_cols),
        ("bin_ohe", build_ohe(), bin_cols),
    ],
    remainder="drop",
)

case1 = run_case(
    "Case 1: cat-encoded + ord-encoded + bin-passthrough",
    preprocessor_c1,
    X_train_base,
    X_test_base,
)
case2 = run_case(
    "Case 2: one-hot + ord-encoded + bin-passthrough",
    preprocessor_c2,
    X_train_base,
    X_test_base,
)
case3 = run_case(
    "Case 3: one-hot + ord-encoded + one-hot",
    preprocessor_c3,
    X_train_base,
    X_test_base,
)


 Skipped missing columns ->
  categorical: []
  ordinal: []
  binary: ['IsYouth']
Case 1: cat-encoded + ord-encoded + bin-passthrough
  X_train: (52062, 12)
  X_test : (13016, 12)
  Positive class ratio (train): 0.119857095
Case 2: one-hot + ord-encoded + bin-passthrough
  X_train: (52062, 67)
  X_test : (13016, 67)
  Positive class ratio (train): 0.119857095
Case 3: one-hot + ord-encoded + one-hot
  X_train: (52062, 73)
  X_test : (13016, 73)
  Positive class ratio (train): 0.119857095


In [49]:
# Train/val split for early stopping for each case (same as Andrew)
def train_val_split(X, y, val_size=0.2):
    return train_test_split(X, y, test_size=val_size, random_state=42, stratify=y)

train_val_splits = {}
for case in [case1, case2, case3]:
    X_tr, X_val, y_tr, y_val = train_val_split(case["X_train_np"], case["y_train_np"])
    train_val_splits[case["name"]] = {
        "X_tr": X_tr,
        "y_tr": y_tr,
        "X_val": X_val,
        "y_val": y_val,}
    
    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
    case["train_loader"] = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    case["val_loader"] = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    

### Neural Network

In [50]:
class MLPBaseline(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)   

def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, patience=PATIENCE):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses)

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_losses.append(loss.item())

        avg_val_loss = np.mean(val_losses)

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break
            
        if best_state is not None:
            model.load_state_dict(best_state)

### Test Set Evaluation

In [51]:
def label_false_positives(X_test, y_test, y_pred):
    fps = np.zeros(y_pred.shape[0])
    y_truth = y_test.copy().reset_index(drop=True)
    test_copy = X_test.copy().reset_index(drop=True)
    for i in test_copy.itertuples():
        if y_pred[i[0]] == 1 and y_truth[i[0]] == 0:
            fps[i[0]] = 1
    test_copy["targetcol"] = np.round(fps, 2)
    return test_copy

def label_false_negatives(X_test, y_test, y_pred):
    fns = np.zeros(y_pred.shape[0])
    y_truth = y_test.copy().reset_index(drop=True)
    test_copy = X_test.copy().reset_index(drop=True)
    for i in test_copy.itertuples():
        if y_pred[i[0]] == 0 and y_truth[i[0]] == 1:
            fns[i[0]] = 1
    test_copy["targetcol"] = np.round(fns, 2)
    return test_copy


def make_loaders(X_train_np, y_train_np, batch_size=512, seed=42, val_size=0.1):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_np, y_train_np, test_size=val_size, random_state=seed, stratify=y_train_np
    )
    train_ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32),
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32),
    )
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False),
    )


def evaluate_model(model, X_test_np, y_test_np):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(X_test_np, dtype=torch.float32, device=DEVICE)
        logits = model(x)            
        logits = logits.reshape(-1)    # robust for [N] or [N,1]
        probs = torch.sigmoid(logits).cpu().numpy()

    y_pred = (probs >= 0.5).astype(int)
    y_true = np.asarray(y_test_np).astype(int).reshape(-1)

    return {
        "y_pred": y_pred,
        "y_prob": probs,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, probs),
    }


In [52]:
results = {}
X_test_for_labels = X_test.copy().reset_index(drop=True)   # raw/readable frame
y_test_series = y_test.copy().reset_index(drop=True)       # for our FP/FN funcs

for case in [case1, case2, case3]:
    print(f"\nTraining {case['name']}...")
    train_loader, val_loader = make_loaders(
        case["X_train_np"], case["y_train_np"], batch_size=BATCH_SIZE, seed=SEED
    )

    model = MLPBaseline(input_dim=case["X_train_np"].shape[1]).to(DEVICE)
    train_model(model, train_loader, val_loader)

    eval_out = evaluate_model(model, case["X_test_np"], case["y_test_np"])
    results[case["name"]] = {
        **eval_out,
        "model": model,
    }

    print(
        f"{case['name']} | Acc={eval_out['accuracy']:.4f} | "
        f"F1={eval_out['f1']:.4f} | ROC-AUC={eval_out['roc_auc']:.4f}"
    )

# metric table
metrics_df = pd.DataFrame([
    {"case": k, "accuracy": v["accuracy"], "f1": v["f1"], "roc_auc": v["roc_auc"]}
    for k, v in results.items()
]).sort_values("roc_auc", ascending=False)
display(metrics_df)

# FP/FN labeled tables by case
fp_tables = {}
fn_tables = {}
for case_name, out in results.items():
    fp_tables[case_name] = label_false_positives(X_test_for_labels, y_test_series, out["y_pred"])
    fn_tables[case_name] = label_false_negatives(X_test_for_labels, y_test_series, out["y_pred"])
    print(case_name, "| FP:", int(fp_tables[case_name]["targetcol"].sum()),
          "| FN:", int(fn_tables[case_name]["targetcol"].sum()))



Training Case 1: cat-encoded + ord-encoded + bin-passthrough...
Epoch 1/40 - Train Loss: 0.3911 - Val Loss: 0.3447
Epoch 2/40 - Train Loss: 0.3429 - Val Loss: 0.3404
Epoch 3/40 - Train Loss: 0.3387 - Val Loss: 0.3371
Epoch 4/40 - Train Loss: 0.3362 - Val Loss: 0.3349
Epoch 5/40 - Train Loss: 0.3332 - Val Loss: 0.3324
Epoch 6/40 - Train Loss: 0.3302 - Val Loss: 0.3314
Epoch 7/40 - Train Loss: 0.3289 - Val Loss: 0.3284
Epoch 8/40 - Train Loss: 0.3268 - Val Loss: 0.3261
Epoch 9/40 - Train Loss: 0.3251 - Val Loss: 0.3265
Epoch 10/40 - Train Loss: 0.3226 - Val Loss: 0.3238
Epoch 11/40 - Train Loss: 0.3218 - Val Loss: 0.3239
Epoch 12/40 - Train Loss: 0.3212 - Val Loss: 0.3229
Epoch 13/40 - Train Loss: 0.3203 - Val Loss: 0.3214
Epoch 14/40 - Train Loss: 0.3186 - Val Loss: 0.3227
Epoch 15/40 - Train Loss: 0.3179 - Val Loss: 0.3215
Epoch 16/40 - Train Loss: 0.3179 - Val Loss: 0.3201
Epoch 17/40 - Train Loss: 0.3160 - Val Loss: 0.3192
Epoch 18/40 - Train Loss: 0.3166 - Val Loss: 0.3189
Epoch 19

,case,accuracy,f1,roc_auc
1,Case 2: one-hot + ord-encoded + bin-passthrough,0.901275,0.470103,0.895853
2,Case 3: one-hot + ord-encoded + one-hot,0.900123,0.442060,0.895748
0,Case 1: cat-encoded + ord-encoded + bin-passth...,0.883221,0.115250,0.811941


Case 1: cat-encoded + ord-encoded + bin-passthrough | FP: 59 | FN: 1461
Case 2: one-hot + ord-encoded + bin-passthrough | FP: 295 | FN: 990
Case 3: one-hot + ord-encoded + one-hot | FP: 255 | FN: 1045


### Disparity by Sex and Race (indepth case-wise analysis)

In [53]:
protected_cols = ["Perceived_Race", "Sex"]

y_true = y_test.reset_index(drop=True).astype(int)
X_grp = X_test.reset_index(drop=True).copy()

def safe_div(a, b):
    return a / b if b != 0 else np.nan

def group_error_table(X_raw, y_true, y_pred, group_col):
    df = pd.DataFrame({
        "group": X_raw[group_col].reset_index(drop=True),
        "y_true": y_true.reset_index(drop=True).astype(int),
        "y_pred": pd.Series(y_pred).reset_index(drop=True).astype(int)
    })

    rows = []
    for g, part in df.groupby("group"):
        yt = part["y_true"].to_numpy()
        yp = part["y_pred"].to_numpy()

        tp = int(((yt == 1) & (yp == 1)).sum())
        tn = int(((yt == 0) & (yp == 0)).sum())
        fp = int(((yt == 0) & (yp == 1)).sum())
        fn = int(((yt == 1) & (yp == 0)).sum())
        n = len(part)

        tpr = safe_div(tp, tp + fn)
        fpr = safe_div(fp, fp + tn)
        fnr = safe_div(fn, tp + fn)
        sel = safe_div(tp + fp, n)

        rows.append({
            "group": g, "count": n, "TP": tp, "TN": tn, "FP": fp, "FN": fn,
            "TPR": tpr, "FPR": fpr, "FNR": fnr, "SelectionRate": sel
        })

    out = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
    return out

def disparity_summary(group_df, metrics=("TPR", "FPR", "FNR", "SelectionRate")):
    d = {}
    for m in metrics:
        vals = group_df[m].dropna()
        d[f"{m}_gap"] = (vals.max() - vals.min()) if len(vals) else np.nan
    # Equalized Odds-style scalar (smaller is better)
    d["EO_gap_max"] = np.nanmax([d["TPR_gap"], d["FPR_gap"]])
    return d

group_tables = {}
summary_rows = []

for case_name, out in results.items():
    y_pred = out["y_pred"]

    for col in protected_cols:
        gtab = group_error_table(X_grp, y_true, y_pred, col)
        group_tables[(case_name, col)] = gtab

        gaps = disparity_summary(gtab)
        summary_rows.append({
            "case": case_name,
            "attribute": col,
            "accuracy": out["accuracy"],
            "f1": out["f1"],
            "roc_auc": out["roc_auc"],
            **gaps
        })

# Detailed per-group tables
for (case_name, col), tbl in group_tables.items():
    print(f"\n=== {case_name} | {col} ===")
    display(tbl)

# Compact comparison table for decision
summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["attribute", "EO_gap_max", "FNR_gap", "FPR_gap", "f1"],
    ascending=[True, True, True, True, False]
).reset_index(drop=True)

print("\n=== Disparity Summary (lower gaps better) ===")
display(summary_df)

#Save this summary table and all the group tables for later analysis and visualization in the next notebook.
summary_df.to_csv(os.path.join(ENCODER_DIR, "disparity_summary.csv"), index=False)

for (case_name, col), tbl in group_tables.items():
    tbl.to_csv(os.path.join(ENCODER_DIR, f"group_table_{case_name}_{col}.csv"), index=False)




=== Case 1: cat-encoded + ord-encoded + bin-passthrough | Perceived_Race ===


,group,count,TP,TN,FP,FN,TPR,FPR,FNR,SelectionRate
0,White,5501,42,4754,24,681,0.058091,0.005023,0.941909,0.011998
1,Black,3464,41,2951,24,448,0.083845,0.008067,0.916155,0.018764
2,Unknown or Legacy,1135,5,1026,1,103,0.046296,0.000974,0.953704,0.005286
3,East/Southeast Asian,839,5,772,3,59,0.078125,0.003871,0.921875,0.009535
4,South Asian,712,4,663,0,45,0.081633,0.000000,0.918367,0.005618
5,Middle-Eastern,623,0,572,4,47,0.000000,0.006944,1.000000,0.006421
6,Indigenous,392,1,335,1,55,0.017857,0.002976,0.982143,0.005102
7,Latino,350,1,324,2,23,0.041667,0.006135,0.958333,0.008571



=== Case 1: cat-encoded + ord-encoded + bin-passthrough | Sex ===


,group,count,TP,TN,FP,FN,TPR,FPR,FNR,SelectionRate
0,M,10477,80,9147,41,1209,0.062064,0.004462,0.937936,0.011549
1,F,2539,19,2250,18,252,0.070111,0.007937,0.929889,0.014573



=== Case 2: one-hot + ord-encoded + bin-passthrough | Perceived_Race ===


...

,group,count,TP,TN,FP,FN,TPR,FPR,FNR,SelectionRate
0,White,5501,245,4671,107,478,0.338866,0.022394,0.661134,0.063988
1,Black,3464,184,2877,98,305,0.376278,0.032941,0.623722,0.081409
2,Unknown or Legacy,1135,30,1009,18,78,0.277778,0.017527,0.722222,0.042291
3,East/Southeast Asian,839,11,764,11,53,0.171875,0.014194,0.828125,0.026222
4,South Asian,712,8,659,4,41,0.163265,0.006033,0.836735,0.016854
5,Middle-Eastern,623,9,572,4,38,0.191489,0.006944,0.808511,0.020867
6,Indigenous,392,23,330,6,33,0.410714,0.017857,0.589286,0.073980
7,Latino,350,5,319,7,19,0.208333,0.021472,0.791667,0.034286



=== Case 3: one-hot + ord-encoded + one-hot | Sex ===


,group,count,TP,TN,FP,FN,TPR,FPR,FNR,SelectionRate
0,M,10477,438,8977,211,851,0.339798,0.022965,0.660202,0.061945
1,F,2539,77,2224,44,194,0.284133,0.019400,0.715867,0.047657



=== Disparity Summary (lower gaps better) ===


,case,attribute,accuracy,f1,roc_auc,TPR_gap,FPR_gap,FNR_gap,SelectionRate_gap,EO_gap_max
0,Case 1: cat-encoded + ord-encoded + bin-passth...,Perceived_Race,0.883221,0.115250,0.811941,0.083845,0.008067,0.083845,0.013662,0.083845
1,Case 3: one-hot + ord-encoded + one-hot,Perceived_Race,0.900123,0.442060,0.895748,0.247449,0.026908,0.247449,0.064555,0.247449
2,Case 2: one-hot + ord-encoded + bin-passthrough,Perceived_Race,0.901275,0.470103,0.895853,0.303571,0.019163,0.303571,0.076244,0.303571
3,Case 1: cat-encoded + ord-encoded + bin-passth...,Sex,0.883221,0.115250,0.811941,0.008047,0.003474,0.008047,0.003024,0.008047
4,Case 3: one-hot + ord-encoded + one-hot,Sex,0.900123,0.442060,0.895748,0.055665,0.003564,0.055665,0.014289,0.055665
5,Case 2: one-hot + ord-encoded + bin-passthrough,Sex,0.901275,0.470103,0.895853,0.058142,0.001978,0.058142,0.010145,0.058142


1. Case 1 is not viable despite the lowest gaps.
- Very high miss rate (FN=1209) and very low F1=0.1152.
- It seems “fairer” mainly because it predicts very few positives

2. Case 2 vs Case 3 (tradeoff):
- Case 2 has slightly better utility (F1 0.47 vs 0.44, ROC 0.8958 vs 0.8957).
- But Case 3 has much better disparity:
   -  Race EO_gap_max: 0.2474 (Case 3) vs 0.3035 (case 2)
   - Sex EO_gap_max: 0.0556 (case 3) vs 0.0581 (case 2)
- That fairness gain is large compared to the small performance drop, so for debiasing work, case 3 is the better choice.


## Explaination Tables

In [54]:
case3_name = next(k for k in results.keys() if k.startswith("Case 3"))
case_map = {c["name"]: c for c in [case1, case2, case3]}

case3_out = results[case3_name]          # contains y_pred, y_prob, metrics, model
case3_cfg = case_map[case3_name]         # contains preprocessor, scaler, X/y arrays

model_case3 = case3_out["model"]
y_pred_case3 = case3_out["y_pred"]

preprocessor_case3 = case3_cfg["preprocessor"]
scaler_case3 = case3_cfg["scaler"]

X_test_case3_raw = X_test.copy().reset_index(drop=True)   # raw test frame for subgroup + DiCE
y_test_case3 = y_test.copy().reset_index(drop=True).astype(int)

print(case3_name)
print("Accuracy:", case3_out["accuracy"], "| F1:", case3_out["f1"], "| ROC-AUC:", case3_out["roc_auc"])
print("FP:", int(((y_pred_case3 == 1) & (y_test_case3.to_numpy() == 0)).sum()))
print("FN:", int(((y_pred_case3 == 0) & (y_test_case3.to_numpy() == 1)).sum()))


Case 3: one-hot + ord-encoded + one-hot
Accuracy: 0.9001229256299939 | F1: 0.44206008583690987 | ROC-AUC: 0.8957481691376594
FP: 255
FN: 1045


In [55]:
exp_mod = globals().get("exptb", globals().get("exptbl"))
if exp_mod is None:
    raise NameError("Neither exptb nor exptbl is available. Import exp as exptb first.")

min_sup = 0.1
group_cols = ["Perceived_Race", "Sex", "Age_group__at_arrest_"]
exp_table_cols = group_cols + ["targetcol"]

In [56]:
#label encoding exp_table columns for subgroup discovery and DiCE
X_train_exp = X_train.copy(deep=True)
X_test_exp = X_test.copy(deep=True)

encoders = {}
for col in group_cols:
    le = LabelEncoder()
    X_train_exp[col] = le.fit_transform(X_train_exp[col].astype(str))
    mapping = {v: i for i, v in enumerate(le.classes_)}
    X_test_exp[col] = X_test_exp[col].astype(str).map(mapping).fillna(-1).astype(int)
    encoders[col] = le
    

In [57]:
# Only FP exp table
x_res_fp_case3 = label_false_positives(X_test_exp, y_test_case3, y_pred_case3)
temp_case3_fp_path = os.path.join(FP_DIR, "temp_case3_fp.csv")
target_case3_fp_path = os.path.join(FP_DIR, "target_case3_fp.csv")
x_res_fp_case3[exp_table_cols].to_csv(temp_case3_fp_path, index=False)
pd.DataFrame().to_csv(target_case3_fp_path, index=False)

res_fp_case3 = exp_mod.calculate_table(
    target_case3_fp_path, temp_case3_fp_path, target_case3_fp_path, min_support_param=min_sup
)


Compiling with commands:  ['g++', '-std=c++17', '/Users/om-college/Work/2 Canada/York/Winter26/Fairness and Bias in AI/groupfairnessproject-local/exp/Explanations.cpp', '/Users/om-college/Work/2 Canada/York/Winter26/Fairness and Bias in AI/groupfairnessproject-local/exp/Lighthouse.cpp', '-o', 'program', '-I/opt/homebrew/opt/boost/include']
Arguments to the program: baseline_nn_output/fp_analysis/target_case3_fp.csv baseline_nn_output/fp_analysis/temp_case3_fp.csv 3 15 0 baseline_nn_output/fp_analysis/target_case3_fp.csv 0.1
Time: 0:00:10.255960


In [58]:
fp_exp_case3 = pd.read_csv(res_fp_case3, sep=";")
# sort in descending order of targetcol 
sorted_fp_case3 = fp_exp_case3.sort_values(by=["targetcol"], ascending=[False])
sorted_fp_case3

,Perceived_Race,Sex,Age_group__at_arrest_,targetcol,support
9,0,1,*,0.030,2864
1,0,*,*,0.028,3464
2,7,*,3,0.027,1511
6,*,1,3,0.025,2601
3,*,*,2,0.024,4237
4,7,*,2,0.024,1687
7,*,*,3,0.024,3210
14,*,1,2,0.024,3378
11,*,1,4,0.021,1479
0,*,*,*,0.020,13016


## DiCE ML

In [59]:
import dice_ml
import recourse_metrics
from raiutils.exceptions import UserConfigValidationException
import json
import ast

In [60]:
def get_test_with_pred_class(X_test, y_pred, num_counterfactuals, target_class, target_column):
    # Concat the predicted values to the data for counterfactual generation
    df_predicted = pd.DataFrame(y_pred, columns=[target_column])
    df_test_pred = pd.concat([X_test.reset_index(drop=True), df_predicted.reset_index(drop=True)], axis=1)
    target_set = df_test_pred.loc[df_test_pred[target_column] == target_class].drop([target_column], axis=1)
    if num_counterfactuals is None:
        num_counterfactuals = target_set.shape[0]
        print(f"setting number of counterfactuals to n={num_counterfactuals}")
    elif num_counterfactuals > target_set.shape[0]:
        num_counterfactuals = target_set.shape[0]
        print(f"warning more counterfactuals requested than num datapoints with label, setting n={num_counterfactuals}")
    target_set = target_set.iloc[0:num_counterfactuals,:].reset_index(drop=True)
    return target_set

# Counterfactual generation func from REACT
def produce_cfs(dataset, cf_method, features_to_perturb, max_cfs_per_sample):
    a = []
    for i in range(dataset.shape[0]):
      sample = dataset.iloc[[i]]
      try:
        cf = cf_method.generate_counterfactuals(sample, 
                                                total_CFs=max_cfs_per_sample, 
                                                desired_class="opposite",
                                                posthoc_sparsity_param=0.1,
                                                posthoc_sparsity_algorithm="linear", 
                                                features_to_vary = features_to_perturb,
                                                verbose=False)
        cfs_list = json.loads(cf.to_json()).get('cfs_list')[0]
        a.append(cfs_list)
      except UserConfigValidationException:
        # no counterfactuals found
        print("no counterfactuals found")
        a.append([])
        continue
    return a

def get_feature_type(feature_name):
    categorical_features = ['Perceived_Race', 'Sex', 'Occurrence_Category', 'ArrestLocDiv']
    ordinal_features = ['Age_group__at_arrest_', 'Arrest_Quarter']
    binary_features = ['IsYouth','Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__',
                  'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Mental_inst',
                  'Actions_at_arrest___Assaulted_o', 'Actions_at_arrest___Cooperative']
    if feature_name in categorical_features:
        return "categorical"
    elif feature_name in ordinal_features:
        return "ordinal"
    else:
        return "binary"
    
def get_feature_details(df, features_perturb):
    feature_details = {}
    for feature in features_perturb:
            feature_type = get_feature_type(feature)
            
            if feature_type == "numerical":
                min_val, max_val = df[feature].min(), df[feature].max()
                feature_details[feature] = {
                    "type": feature_type,
                    "range_min": min_val,
                    "range_max": max_val
                }

            if feature_type == "ordinal":
                feature_details[feature] = {
                    "type": feature_type,
                    "range_min": df[feature].min(),
                    "range_max": df[feature].max()
                }

            if feature_type == "categorical":
                all_categ_values = sorted(df[feature].unique().tolist())
                feature_details[feature] = {
                    "type": feature_type,
                    "categ_values": all_categ_values
                }
                
            # add binary handling later if needed (currently treating as ordinal with range [0,1])
    return feature_details


In [61]:
# Parameters for counterfacutal testing
max_cfs = 10
max_counterfactuals = 1000 #None
target_class = 0 # want to find samples with this classifiaction to find counterfactual of opposite "desired" class
target_column = 'StripSearch'
features_perturb =  ['ArrestLocDiv','Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__',
                  'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Mental_inst',
                  'Actions_at_arrest___Assaulted_o', 'Actions_at_arrest___Cooperative']
min_sup = 0.1 # Minimum support for explanation tables
# explain_features =  ['Perceived_Race', 'Sex', 'Age_group__at_arrest_']
# group cols = explain_features

In [62]:
target_set_case3 = get_test_with_pred_class(
    X_test_case3_raw, y_pred_case3, max_counterfactuals, target_class, target_column
)


In [63]:
class NNCase3Wrapper:
    def __init__(self, nn_model, preprocessor, scaler, feature_columns, device=DEVICE):
        self.nn_model = nn_model
        self.preprocessor = preprocessor
        self.scaler = scaler
        self.feature_columns = feature_columns
        self.device = device

    def _to_model_features(self, X):
        if isinstance(X, pd.DataFrame):
            X_df = X.copy()
        else:
            X_df = pd.DataFrame(X, columns=self.feature_columns)

        X_df = X_df[self.feature_columns]
        X_enc = self.preprocessor.transform(X_df)
        if hasattr(X_enc, "toarray"):
            X_enc = X_enc.toarray()
        X_scaled = self.scaler.transform(X_enc).astype(np.float32)
        return X_scaled

    def predict_proba(self, X):
        X_np = self._to_model_features(X)
        with torch.no_grad():
            xb = torch.tensor(X_np, dtype=torch.float32, device=self.device)
            probs = torch.sigmoid(self.nn_model(xb)).cpu().numpy().reshape(-1, 1)
        return np.hstack([1.0 - probs, probs])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [64]:
nn_wrapper_case3 = NNCase3Wrapper(
    nn_model=model_case3,
    preprocessor=preprocessor_case3,
    scaler=scaler_case3,
    feature_columns=X_test_case3_raw.columns.tolist(),
    device=DEVICE,
)

d = dice_ml.Data(
    dataframe=pd.concat([X_test_case3_raw.reset_index(drop=True), y_test_case3.reset_index(drop=True)], axis=1),
    continuous_features=[],
    outcome_name=target_column
)
m = dice_ml.Model(model=nn_wrapper_case3, backend="sklearn")
exp = dice_ml.Dice(d, m, method="random")

cfs_out = produce_cfs(target_set_case3, exp, features_perturb, max_cfs)
target_set_case3["cfs_list"] = cfs_out
cfs_case3_path = os.path.join(DICE_DIR, "cfs_case3.csv")
target_set_case3.to_csv(cfs_case3_path, index=False)

print("rows:", len(target_set_case3))
print("rows with >=1 CF:", int((target_set_case3["cfs_list"].apply(len) > 0).sum())) 


100%|██████████| 1/1 [00:00<00:00,  9.94it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
no counterfactuals found


100%|██████████| 1/1 [00:00<00:00, 10.72it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
no counterfactuals found


100%|██████████| 1/1 [00:00<00:00,  9.85it/s]


...

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
no counterfactuals found


100%|██████████| 1/1 [00:00<00:00, 11.04it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
no counterfactuals found


100%|██████████| 1/1 [00:00<00:00, 14.63it/s]

rows: 1000
rows with >=1 CF: 403


### Recourse metrics

In [65]:
cfs_case3_path = os.path.join(DICE_DIR, "cfs_case3.csv")
df = pd.read_csv(cfs_case3_path)
df['cfs_list'] = df['cfs_list'].apply(ast.literal_eval)
df['cfs_list'] = df['cfs_list'].apply(lambda x: np.array(x))

def encode_group_cols_for_exp(df_in, ref_df, cols): #Just for post-hoc explanation-table tool compatibility, not a general encoding function.
    out = df_in.copy()
    for c in cols:
        le = LabelEncoder()
        le.fit(ref_df[c].astype(str))  # fit on train/reference
        mapping = {v: i for i, v in enumerate(le.classes_)}
        out[c] = out[c].astype(str).map(mapping).fillna(-1).astype(int)
    return out

df = df[group_cols + features_perturb + ['cfs_list']]
selected_features = features_perturb
feature_details = get_feature_details(df, selected_features)

# Compute and save recourse metrics for the generated counterfactuals
r_cost_df = recourse_metrics.recourse_cost_df(df, feature_details, 1)
r_cost_to_summarize = r_cost_df[group_cols+['recourse_cost']].copy()
r_cost_to_summarize_enc = encode_group_cols_for_exp(r_cost_to_summarize, X_train, group_cols)
temp_cost_case3_path = os.path.join(RECOURSE_TEMP_DIR, "temp_cost_case3.csv")
r_cost_to_summarize_enc.to_csv(temp_cost_case3_path, index=False)

r_avail_df = recourse_metrics.recourse_availability_df(df)
r_avail_df_to_summarize = r_avail_df[group_cols+['recourse_availability']].copy()
r_avail_df_to_summarize_enc = encode_group_cols_for_exp(r_avail_df_to_summarize, X_train, group_cols)
temp_avail_case3_path = os.path.join(RECOURSE_TEMP_DIR, "temp_avail_case3.csv")
r_avail_df_to_summarize_enc.to_csv(temp_avail_case3_path, index=False)

r_choice_df = recourse_metrics.recourse_choice_df(df)
r_choice_df_to_summarize = r_choice_df[group_cols+['recourse_choice']].copy()
r_choice_df_to_summarize_enc = encode_group_cols_for_exp(r_choice_df_to_summarize, X_train, group_cols)
temp_choice_case3_path = os.path.join(RECOURSE_TEMP_DIR, "temp_choice_case3.csv")
r_choice_df_to_summarize_enc.to_csv(temp_choice_case3_path, index=False)


In [66]:
avail_table_case3_path = os.path.join(RECOURSE_TABLE_DIR, "avail_table_case3.csv")
temp_avail_case3_path = os.path.join(RECOURSE_TEMP_DIR, "temp_avail_case3.csv")
empty_df = pd.DataFrame().to_csv(avail_table_case3_path, index=False)
# explanation tables for recourse availability 
res_nn = exp_mod.calculate_table(
    avail_table_case3_path, temp_avail_case3_path, avail_table_case3_path, min_support_param=min_sup
)
exp_table = pd.read_csv(res_nn, sep=";")
exp_table


Compiling with commands:  ['g++', '-std=c++17', '/Users/om-college/Work/2 Canada/York/Winter26/Fairness and Bias in AI/groupfairnessproject-local/exp/Explanations.cpp', '/Users/om-college/Work/2 Canada/York/Winter26/Fairness and Bias in AI/groupfairnessproject-local/exp/Lighthouse.cpp', '-o', 'program', '-I/opt/homebrew/opt/boost/include']
Arguments to the program: baseline_nn_output/recourse_tables/avail_table_case3.csv baseline_nn_output/recourse_temp/temp_avail_case3.csv 3 15 0 baseline_nn_output/recourse_tables/avail_table_case3.csv 0.1
Time: 0:00:05.686907


,Perceived_Race,Sex,Age_group__at_arrest_,recourse_availability,support
0,*,*,*,0.403,1000
1,*,1,2,0.410,266
2,*,1,*,0.403,808
3,0,*,*,0.438,292
4,*,*,1,0.494,158
5,7,*,2,0.457,129
6,0,1,*,0.435,246
7,*,1,1,0.508,132
8,*,*,3,0.376,242
9,7,1,*,0.399,316


In [67]:
choice_table_case3_path = os.path.join(RECOURSE_TABLE_DIR, "choice_table_case3.csv")
temp_choice_case3_path = os.path.join(RECOURSE_TEMP_DIR, "temp_choice_case3.csv")
empty_df = pd.DataFrame().to_csv(choice_table_case3_path, index=False)
res_nn = exp_mod.calculate_table(
    choice_table_case3_path, temp_choice_case3_path, choice_table_case3_path, min_support_param=min_sup
)
exp_table = pd.read_csv(res_nn, sep=";")
exp_table


Compiling with commands:  ['g++', '-std=c++17', '/Users/om-college/Work/2 Canada/York/Winter26/Fairness and Bias in AI/groupfairnessproject-local/exp/Explanations.cpp', '/Users/om-college/Work/2 Canada/York/Winter26/Fairness and Bias in AI/groupfairnessproject-local/exp/Lighthouse.cpp', '-o', 'program', '-I/opt/homebrew/opt/boost/include']
Arguments to the program: baseline_nn_output/recourse_tables/choice_table_case3.csv baseline_nn_output/recourse_temp/temp_choice_case3.csv 3 15 0 baseline_nn_output/recourse_tables/choice_table_case3.csv 0.1
Time: 0:00:05.561304


,Perceived_Race,Sex,Age_group__at_arrest_,recourse_choice,support
0,*,*,*,3.822,1000
1,0,1,*,4.167,246
2,7,1,*,3.851,316
3,*,1,1,4.803,132
4,7,*,2,4.310,129
5,0,*,*,4.205,292
6,*,1,3,3.767,189
7,*,1,2,4.023,266
8,*,0,*,3.594,192
9,*,*,4,3.385,117


In [68]:
cost_table_case3_path = os.path.join(RECOURSE_TABLE_DIR, "cost_table_case3.csv")
temp_cost_case3_path = os.path.join(RECOURSE_TEMP_DIR, "temp_cost_case3.csv")
empty_df = pd.DataFrame().to_csv(cost_table_case3_path, index=False)
res_nn = exp_mod.calculate_table(
    cost_table_case3_path, temp_cost_case3_path, cost_table_case3_path, min_support_param=min_sup
)
exp_table = pd.read_csv(res_nn, sep=";")
exp_table


Compiling with commands:  ['g++', '-std=c++17', '/Users/om-college/Work/2 Canada/York/Winter26/Fairness and Bias in AI/groupfairnessproject-local/exp/Explanations.cpp', '/Users/om-college/Work/2 Canada/York/Winter26/Fairness and Bias in AI/groupfairnessproject-local/exp/Lighthouse.cpp', '-o', 'program', '-I/opt/homebrew/opt/boost/include']
Arguments to the program: baseline_nn_output/recourse_tables/cost_table_case3.csv baseline_nn_output/recourse_temp/temp_cost_case3.csv 3 15 0 baseline_nn_output/recourse_tables/cost_table_case3.csv 0.1
Time: 0:00:05.283045


,Perceived_Race,Sex,Age_group__at_arrest_,recourse_cost,support
0,*,*,*,1.0,403
